<a href="https://colab.research.google.com/github/zhangyuwen193-cell/114-2-Programing-Language/blob/main/HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# 0. 安裝套件
# =========================
!pip -q install gspread gspread-dataframe beautifulsoup4 requests pandas sentence-transformers faiss-cpu google-genai


# =========================
# 1. 匯入套件
# =========================
import re
import time
import requests
import pandas as pd
import faiss

from datetime import datetime
from urllib.parse import urljoin
from bs4 import BeautifulSoup

import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.colab import auth, userdata
from google.auth import default

from sentence_transformers import SentenceTransformer

from google import genai


# =========================
# 2. Google Sheet 設定
# =========================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 改成你的 Google Sheet 網址
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Y5AdONl_vvSWPuiVS6lsPdz5DgayWJAuL6gNWhoRe58/edit?usp=sharing"

sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")


# =========================
# 3. 基本設定
# =========================
PTT_WORKSHEET_NAME = "PTT_movie"

PTT_HEADER = [
    "post_id",
    "title",
    "url",
    "date",
    "author",
    "nrec",
    "created_at",
    "fetched_at",
    "content",
]

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"


# =========================
# 4. 建立 requests session，避免 PTT 連線中斷
# =========================
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)


def get_soup(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "zh-TW,zh;q=0.9,en;q=0.8",
        "Referer": "https://www.ptt.cc/bbs/movie/index.html",
    }

    cookies = {
        "over18": "1"
    }

    resp = session.get(
        url,
        timeout=30,
        headers=headers,
        cookies=cookies
    )

    resp.raise_for_status()

    return BeautifulSoup(resp.text, "html.parser")


# =========================
# 5. Google Sheet 工具函式
# =========================
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(
            title=title,
            rows=str(rows),
            cols=str(len(header) + 5)
        )
        ws.update([header])
        return ws

    values = ws.get_all_values()

    if not values:
        ws.update([header])
    elif values[0] != header:
        print("⚠️ 表頭不一致，將清空工作表並重建表頭。")
        ws.clear()
        ws.update([header])

    return ws


def read_sheet_df(ws, header):
    df = get_as_dataframe(
        ws,
        evaluate_formulas=True,
        dtype=str
    ).dropna(how="all")

    if df.empty:
        return pd.DataFrame(columns=header)

    df = df.loc[:, [
        c for c in df.columns
        if not str(c).startswith("Unnamed")
    ]]

    for col in header:
        if col not in df.columns:
            df[col] = ""

    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    df_out = df.copy()

    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""

    df_out = df_out[header].infer_objects(copy=False).fillna("")

    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()

    set_with_dataframe(
        ws,
        df_out,
        include_index=False,
        include_column_header=True,
        resize=True
    )

    return len(df_out)


ws_ptt = ensure_worksheet(
    sh,
    PTT_WORKSHEET_NAME,
    PTT_HEADER
)

print(f"✅ 已準備 worksheet：{ws_ptt.title}")


# =========================
# 6. PTT 爬蟲函式
# =========================
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None

    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0

    txt = nrec_span.get_text(strip=True)

    if txt == "爆":
        return 100

    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10

    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []

    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")

        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))

        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })

    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")

    if not main:
        return "", ""

    created_at = ""

    metalines = main.select("div.article-metaline")

    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")

        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    for node in main.select(
        "div.article-metaline, div.article-metaline-right, div.push"
    ):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text, created_at


def make_post_id(url):
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=1, delay=3.0):
    all_rows = []
    index_url = PTT_MOVIE_INDEX

    for page in range(int(pages)):
        try:
            print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
            index_soup = get_soup(index_url)
        except Exception as e:
            print(f"❌ 列表頁讀取失敗：{e}")
            break

        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)

                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }

                all_rows.append(row)
                time.sleep(delay)

            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")
                time.sleep(delay)

        prev_url = get_prev_index_url(index_soup)

        if not prev_url:
            break

        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)

    print(f"✅ 本次爬到 {len(df)} 篇文章")

    return df


# =========================
# 7. 執行爬蟲並寫入 Google Sheet
# =========================
new_posts_df = crawl_ptt_movie(
    pages=1,
    delay=3.0
)

old_posts_df = read_sheet_df(
    ws_ptt,
    PTT_HEADER
)

print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat(
    [old_posts_df, new_posts_df],
    ignore_index=True
)

ptt_posts_df = ptt_posts_df.drop_duplicates(
    subset=["post_id"],
    keep="last"
)

ptt_posts_df = ptt_posts_df.sort_values(
    by="fetched_at",
    ascending=False
)

written_count = write_sheet_df(
    ws_ptt,
    ptt_posts_df,
    PTT_HEADER
)

print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(
    ws_ptt,
    PTT_HEADER
)

print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")


# =========================
# 8. 從 Google Sheet 建立 RAG 資料
# =========================
rag_source_df = read_sheet_df(
    ws_ptt,
    PTT_HEADER
)

rag_source_df = rag_source_df[
    rag_source_df["content"].astype(str).str.strip() != ""
].copy()

print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")


# =========================
# 9. 建立 Embedding 與 FAISS 索引
# =========================
print("正在載入多語言 Embedding 模型...")

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []

    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (
            f"標題：{title}\n"
            f"作者：{author}\n"
            f"日期：{date}\n"
            f"推文數：{nrec}\n"
            f"內容：{content}"
        )

        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })

    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(
    f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，"
    f"向量維度 {rag_embeddings.shape[1]}"
)


# =========================
# 10. Gemini 新版 google.genai 設定
# =========================
api_key = userdata.get("gemini")

if not api_key:
    raise ValueError(
        "找不到 Colab Secret：gemini。"
        "請先在 Colab 左側 Secrets 新增 Gemini API key，名稱要叫 gemini。"
    )

client = genai.Client(api_key=api_key)

GEMINI_MODEL_NAME = "gemini-2.0-flash"

print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


# =========================
# 11. RAG 檢索與問答
# =========================
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []

    q_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = rag_index.search(q_emb, int(k))

    results = []

    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue

        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)

    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)

    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join([
        f"來源標題：{d['title']}\n"
        f"來源網址：{d['url']}\n"
        f"{d['text']}"
        for d in docs
    ])

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。

請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文。
回答最後請列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=prompt
    )

    return response.text


# =========================
# 12. 測試問答
# =========================
question = input("請輸入問題：")

answer = query_rag(
    question,
    k=3
)

print(answer)

✅ 已開啟試算表：程式語言作業範例測試資料 的副本
✅ 已準備 worksheet：PTT_movie
📄 正在讀取列表頁 1/1: https://www.ptt.cc/bbs/movie/index.html
✅ 本次爬到 20 篇文章
📌 Google Sheet 原本有 0 筆
✅ 已寫入 Google Sheet：20 筆
🔍 從 Google Sheet 重新讀回：20 筆
✅ 寫入驗證成功
📚 可用於 RAG 的文章數：20
正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding 模型載入完成


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RAG 索引建立完成：20 篇文章，向量維度 384
✅ Gemini 已設定：gemini-2.0-flash
請輸入問題：您好


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 59.877242795s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go